# CS781 Phase 3 - build the submission files (CPU only)
Builds **20 submission files** from the six run bundles (`bundle_A.zip`, `bundle_B.zip`, `bundle_C.zip`, `bundle_D.zip`,
`bundle_E.zip`, `bundle_extra.zip`). Nothing is trained: it only reads the saved sentence probabilities.

- **Single runs**: the run's own `prediction.json` (exactly what the run wrote).
- **Ensembles**: average of the members' sentence probabilities, then argmax. Members are chosen on dev only.
- **Dissent-aware ensembles**: averaging never predicts the rare label *Dissent* (no ensemble did on test), while some single
  runs do. Here a sentence becomes *Dissent* when at least `k` members predict it. Dev has no Dissent sentences, so dev can
  only show the cost of this rule (sentences wrongly turned into Dissent), not its gain.

Every file is validated (all test documents, one label per sentence, valid labels), and its dev version is scored with the
official evaluator. Files are numbered in submission order: `submissions/01_<name>.zip` ... plus `SUBMISSIONS.csv`
(what each file is) and `all_submissions.zip` (everything in one download).

**Inputs:** one dataset with the six bundles + `cs781-p3-code`. Internet ON, secret `HF_TOKEN`, **Accelerator: None**.
Save & Run All (~2-4 min). Edit the plan cell to change what gets built.

In [ ]:
import os, sys, json, re, glob, shutil, subprocess, zipfile
W = "/kaggle/working"
WORK, P3DATA, RUNS = f"{W}/p3code", f"{W}/p3data", f"{W}/runs"
def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("rr_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
os.makedirs(P3DATA, exist_ok=True)
for f in list_repo_files(REPO, repo_type="dataset", token=tok_):
    if f.startswith("Phase_3") and f.endswith(".json"):
        shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), os.path.join(P3DATA, os.path.basename(f)))
import rr_data
for dom in ("CL", "IT"):
    json.dump({x["id"]: x["labels"] for x in rr_data.load_split(f"{P3DATA}/{dom}_dev.json")}, open(f"{W}/dev_truth_{dom}.json", "w"))
# ---- collect runs from every bundle attached (zip files, or folders Kaggle already unzipped)
os.makedirs(RUNS, exist_ok=True)
for z in glob.glob("/kaggle/input/**/bundle_*.zip", recursive=True):
    zipfile.ZipFile(z).extractall(RUNS)
for p in glob.glob("/kaggle/input/**/test_prob.npz", recursive=True):
    d = os.path.dirname(p); t = os.path.basename(d)
    if re.match(r"^[A-E]\d\d_", t) and not os.path.exists(f"{RUNS}/{t}"):
        shutil.copytree(d, f"{RUNS}/{t}")
FOUND = sorted(t for t in os.listdir(RUNS) if os.path.exists(f"{RUNS}/{t}/dev_prob.npz") and not t.startswith("ens_"))
print(len(FOUND), "runs:", FOUND)

## Plan (edit here)

In [ ]:
N_SUBMISSIONS = 20
MIN_DEV = 0.40          # runs below this dev macro-F1 (collapsed) are never ensemble members

import numpy as np
import rr_data as D
labels = D.load_vocab(f"{P3DATA}/label_vocab.json")
DIS = labels.index("Dissent")

def summary(t):
    return json.load(open(f"{RUNS}/{t}/summary.json"))
def dev_f1(t):
    return summary(t)["dev"]["macro_f1"]

POOL = [t for t in FOUND if dev_f1(t) >= MIN_DEV]
ranked = sorted(POOL, key=dev_f1, reverse=True)
per = {m: [t for t in ranked if t.startswith(m)] for m in "ABCDE"}
best = lambda m: per[m][0]
ENSEMBLES = {                                                     # name -> members (chosen on dev only)
    "ens_top3permodel":     [t for v in per.values() for t in v[:3]],
    "ens_top10":            ranked[:10],
    "ens_noA_top3permodel": [t for m, v in per.items() if m != "A" for t in v[:3]],
    "ens_top20":            ranked[:20],
    "ens_top5":             ranked[:5],
    "ens_bestpermodel":     [v[0] for v in per.values() if v],
    "ens_top2permodel":     [t for v in per.values() for t in v[:2]],
    "ens_top5permodel":     [t for v in per.values() for t in v[:5]],
    "ens_top15":            ranked[:15],
    "ens_all":              ranked,
}

# (file name, members, Dissent votes k (0 = off), why)  -- in submission order; backups fill in for duplicates
PLAN = [
    ("ens_top3permodel",          ENSEMBLES["ens_top3permodel"], 0, "best dev ensemble: top 3 runs of each model"),
    ("ens_top10",                 ENSEMBLES["ens_top10"], 0, "10 best runs overall"),
    ("ens_noA_top3permodel",      ENSEMBLES["ens_noA_top3permodel"], 0, "top 3 of B-E (weakest model left out)"),
    ("ens_top3permodel_dissent2", ENSEMBLES["ens_top3permodel"], 2, "best ensemble + Dissent where >= 2 members say so"),
    ("ens_top3permodel_dissent1", ENSEMBLES["ens_top3permodel"], 1, "best ensemble + Dissent where any member says so"),
    ("B24_lora_seed7",            ["B24_lora_seed7"], 0, "Dissent probe: strong single run that predicts Dissent"),
    (best("D"),                   [best("D")], 0, "best single run of D"),
    (best("E"),                   [best("E")], 0, "best single run of E"),
    (best("C"),                   [best("C")], 0, "best single run of C"),
    (best("A"),                   [best("A")], 0, "best single run of A"),
    ("D03_lorma_lr2e4",           ["D03_lorma_lr2e4"], 0, "best LoRMA run (compulsory technique)"),
    ("ens_top20",                 ENSEMBLES["ens_top20"], 0, "20 best runs overall"),
    ("ens_top5",                  ENSEMBLES["ens_top5"], 0, "5 best runs overall"),
    ("ens_bestpermodel",          ENSEMBLES["ens_bestpermodel"], 0, "best run of each model"),
    ("ens_top2permodel",          ENSEMBLES["ens_top2permodel"], 0, "top 2 runs of each model"),
    ("ens_top5permodel",          ENSEMBLES["ens_top5permodel"], 0, "top 5 runs of each model"),
    ("ens_noA_top3permodel_dissent2", ENSEMBLES["ens_noA_top3permodel"], 2, "B-E ensemble + Dissent (>= 2 members)"),
    ("ens_top10_dissent1",        ENSEMBLES["ens_top10"], 1, "top-10 ensemble + Dissent (any member)"),
    ("E06_adalora",               ["E06_adalora"], 0, "best AdaLoRA run"),
    ("D07_qlora",                 ["D07_qlora"], 0, "best-model QLoRA run (4-bit backbone)"),
    # ---- backups (used only if something above is a duplicate of an earlier file or missing)
    ("ens_top15",                 ENSEMBLES["ens_top15"], 0, "backup: 15 best runs"),
    ("ens_top5permodel_dissent2", ENSEMBLES["ens_top5permodel"], 2, "backup: top-5-per-model + Dissent (>= 2)"),
    ("ens_all",                   ENSEMBLES["ens_all"], 0, "backup: every non-collapsed run"),
    ("D01_lora",                  ["D01_lora"], 0, "backup: base model on D"),
]
for name, mem, k, why in PLAN:
    print(f"{name:32s} k={k}  {len(mem):3d} members  | {why}")

## Build, check, score on dev

In [ ]:
import csv, io
def probs(t, split):
    f = np.load(f"{RUNS}/{t}/{split}_prob.npz")
    return {str(k): f[f"p{i}"].astype(np.float32) for i, k in enumerate(f["ids"])}
CACHE = {}
def P(t, split):
    if (t, split) not in CACHE:
        CACHE[(t, split)] = probs(t, split)
    return CACHE[(t, split)]

def predict(members, split, k=0):
    """Average of members' probabilities -> argmax; with k > 0, Dissent wherever >= k members' argmax is Dissent."""
    out, n_dis = {}, 0
    for doc in P(members[0], split):
        stack = np.stack([P(t, split)[doc] for t in members])         # (members, sentences, labels)
        y = stack.mean(0).argmax(1)
        if k:
            votes = (stack.argmax(2) == DIS).sum(0)
            n_dis += int(((votes >= k) & (y != DIS)).sum())
            y = np.where(votes >= k, DIS, y)
        out[doc] = [labels[i] for i in y]
    return out, n_dis

test_docs = [D.RRDoc(x["id"], D.DOMAINS.index(dm), x["text"]) for dm in D.DOMAINS
             for x in D.load_split(D.split_path(P3DATA, dm, "test"))]
def official(pred):
    p = f"{W}/_dev_pred.json"; json.dump(pred, open(p, "w"))
    r = subprocess.run(f"python evaluate_phase3.py --truth-cl {W}/dev_truth_CL.json --truth-it {W}/dev_truth_IT.json "
                       f"--prediction {p} --labels {P3DATA}/label_vocab.json", shell=True, capture_output=True, text=True, cwd=WORK)
    m = re.search(r"Macro-F1:\s+([0-9.]+)", r.stdout)
    return float(m.group(1)) if m else float("nan")

SUB = f"{W}/submissions"
shutil.rmtree(SUB, ignore_errors=True); os.makedirs(SUB)
rows, seen, i = [], {}, 0
for name, members, k, why in PLAN:
    if i == N_SUBMISSIONS:
        break
    missing = [t for t in members if t not in FOUND]
    if not members or missing:
        print(f"skip {name}: missing runs {missing}"); continue
    test, n_dis_test = predict(members, "test", k)
    single = len(members) == 1 and not k
    if single:                                                        # single run: use exactly its own files
        own = json.load(open(f"{RUNS}/{members[0]}/prediction.json"))
        diff = sum(a != b for d in own for a, b in zip(own[d], test[d]))
        if diff: print(f"note {name}: {diff} sentences differ between saved probabilities and prediction.json (rounding); using prediction.json")
        test = own
    key = json.dumps(test, sort_keys=True)
    if key in seen:
        print(f"skip {name}: identical to {seen[key]}"); continue
    D.validate_submission(test, test_docs, labels)                    # every doc, one valid label per sentence
    dev, n_dis_dev = predict(members, "dev", k)
    if single:
        dev = json.load(open(f"{RUNS}/{members[0]}/dev_pred.json"))
    i += 1
    seen[key] = name
    fn = f"{i:02d}_{name}.zip"
    with zipfile.ZipFile(f"{SUB}/{fn}", "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("prediction.json", json.dumps(test, ensure_ascii=False))
    cnt = {l: sum(v.count(l) for v in test.values()) for l in labels}
    rows.append(dict(no=i, file=fn, kind="single" if len(members) == 1 else "ensemble", dissent_votes=k,
                     dev_macro_f1=round(official(dev), 4), n_members=len(members),
                     test_dissent=cnt["Dissent"], test_prec_overruled=cnt["PrecedentOverruled"],
                     test_ruling_present=cnt["RulingByPresentCourt"], dev_sents_set_to_dissent=n_dis_dev,
                     why=why, members=" ".join(members)))

with open(f"{W}/SUBMISSIONS.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
with zipfile.ZipFile(f"{W}/all_submissions.zip", "w") as z:
    for r in rows:
        z.write(f"{SUB}/{r['file']}", arcname=r["file"])
    z.write(f"{W}/SUBMISSIONS.csv", arcname="SUBMISSIONS.csv")

print(f"\n=== SUBMISSIONS ({len(rows)}) === dev = official evaluator on the same rule applied to dev "
      "(max 0.846: dev has no Dissent / PrecedentOverruled)")
print(f"{'no':>2} {'file':40s} {'dev':>6} {'mem':>4} {'Dis':>4} {'dev->Dis':>8}  why")
for r in rows:
    print(f"{r['no']:2d} {r['file']:40s} {r['dev_macro_f1']:6.4f} {r['n_members']:4d} {r['test_dissent']:4d} "
          f"{r['dev_sents_set_to_dissent']:8d}  {r['why']}")
print(f"\nDis = test sentences labelled Dissent; dev->Dis = dev sentences the Dissent rule changed (each one is a dev error)")
print(f"files: {SUB}/  +  {W}/SUBMISSIONS.csv  +  {W}/all_submissions.zip")